## Reorganization the table to adapt the data Mart

In [10]:
#Reorganization the csv to adapt the database - FOREST

import pandas as pd

df = pd.read_csv("filtered data/forest-cover-new1【year-forest.csv")


df_long = df.melt(id_vars=["Country Name"], 
                  var_name="Year", 
                  value_name="Forest Area")

df_long["Year"] = df_long["Year"].str.replace("Forest Area ", "")

df_long.columns = ["Country", "Year", "Forest Area"]



In [11]:
#Reorganization the csv to adapt the database - Polulation
import pandas as pd


forest_df = pd.read_csv("filtered data/forest-cover-v1.csv")
population_df = pd.read_csv("filtered data/GlobalPM25-1998-2022.csv")


population_cleaned = population_df.rename(columns={
    'Country': 'country',
    'Year': 'year',
    'Total Population [million people]': 'total_Population_million_people',
    'Population Coverage [%]': 'population_coverage_perc',
    'Geographic Coverage [%]': 'geographic_coverage_perc'
})


forest_long = forest_df.melt(
    id_vars=['Country Name', 'Population Density (per km²)', 'Population Rank'],
    value_vars=[col for col in forest_df.columns if col.startswith("Forest Area")],
    var_name='year',
    value_name='forest_area'
)


forest_long['year'] = forest_long['year'].str.extract('(\d{4})')


forest_long = forest_long.rename(columns={
    'Country Name': 'country',
    'Population Density (per km²)': 'population_density',
    'Population Rank': 'population_rank'
})


population_cleaned['year'] = population_cleaned['year'].astype(str)
forest_long['year'] = forest_long['year'].astype(str)


merged_df = pd.merge(population_cleaned, forest_long, on=['country', 'year'], how='inner')



final_df = merged_df[[
    'country',
    'total_Population_million_people',
    'population_density',
    'population_rank',
    'population_coverage_perc',
    'geographic_coverage_perc',
    'year'
]]



<>:26: SyntaxWarning: invalid escape sequence '\d'
<>:26: SyntaxWarning: invalid escape sequence '\d'
C:\Users\82472\AppData\Local\Temp\ipykernel_41784\2979164235.py:26: SyntaxWarning: invalid escape sequence '\d'
  forest_long['year'] = forest_long['year'].str.extract('(\d{4})')


## Data fusion 

In [ ]:
# 1 - Country 
import pandas as pd

forest_df = pd.read_csv("filtered data/forest-cover-v1.csv")
population_df = pd.read_csv("filtered data/GlobalPM25-1998-2022.csv")

forest_countries = set(forest_df['Country'].unique())
population_countries = set(population_df['Country'].unique())

all_countries = sorted(forest_countries.union(population_countries))
mapping_df = pd.DataFrame({
    'original_name': all_countries,
    'standard_name': all_countries
})
mapping_df.to_csv("filtered data/country_mapping.csv", index=False)

mapping_df = pd.read_csv("filtered data/country_mapping.csv")


forest_df = forest_df.merge(mapping_df, how='left', left_on='Country', right_on='original_name')
forest_df['Country'] = forest_df['standard_name'].combine_first(forest_df['Country'])
forest_df = forest_df.drop(columns=['original_name', 'standard_name'])

forest_df.to_csv("filtered data/forest-cover-v1-standardized.csv", index=False)


def standardize_country(df, country_column, mapping_df):
    df = df.merge(mapping_df, how='left', left_on=country_column, right_on='original_name')
    df[country_column] = df['standard_name'].combine_first(df[country_column])
    df = df.drop(columns=['original_name', 'standard_name'])
    return df



# 1. population_df
population_df = standardize_country(population_df, 'Country', mapping_df)
population_df.to_csv("filtered data/GlobalPM25-1998-2022-standardized.csv", index=False)

# 2. GlobalWeatherRepository.csv
weather_df = pd.read_csv("filtered data/GlobalWeatherRepository.csv")
weather_df = standardize_country(weather_df, 'Country', mapping_df)
weather_df.to_csv("filtered data/GlobalWeatherRepository-standardized.csv", index=False)

# 3. trafficlist_forcountry.csv
traffic_df = pd.read_csv("filtered data/trafficlist_forcountry.csv")
traffic_df = standardize_country(traffic_df, 'Country', mapping_df)
traffic_df.to_csv("filtered data/trafficlist_forcountry-standardized.csv", index=False)

# 4. air_quality_city.csv
air_df = pd.read_csv("filtered data/air_quality_city.csv")
air_df = standardize_country(air_df, 'Country', mapping_df)
air_df.to_csv("filtered data/air_quality_city-standardized.csv", index=False)




In [ ]:
# 2. GDP - 0 
import pandas as pd

gdp_df = pd.read_csv("Individual challenge-52915/GDP.csv")

gdp_df = gdp_df[(gdp_df['GDP per capita'] != 0) & (~gdp_df['GDP per capita'].isna())]

gdp_df.to_csv("Individual challenge-52915/GDP.csv", index=False)


In [ ]:
# 3.Polulations

import pandas as pd


forest_df = pd.read_csv("filtered data/forest-cover-v1.csv")
population_df = pd.read_csv("filtered data/GlobalPM25-1998-2022.csv")


population_cleaned = population_df.rename(columns={
    'Country': 'country',
    'Year': 'year',
    'Total Population [million people]': 'total_Population_million_people',
    'Population Coverage [%]': 'population_coverage_perc',
    'Geographic Coverage [%]': 'geographic_coverage_perc'
})


forest_long = forest_df.melt(
    id_vars=['Country Name', 'Population Density (per km²)', 'Population Rank'],
    value_vars=[col for col in forest_df.columns if col.startswith("Forest Area")],
    var_name='year',
    value_name='forest_area'
)


forest_long['year'] = forest_long['year'].str.extract('(\d{4})')


forest_long = forest_long.rename(columns={
    'Country Name': 'country',
    'Population Density (per km²)': 'population_density',
    'Population Rank': 'population_rank'
})


population_cleaned['year'] = population_cleaned['year'].astype(str)
forest_long['year'] = forest_long['year'].astype(str)


merged_df = pd.merge(population_cleaned, forest_long, on=['country', 'year'], how='inner')



final_df = merged_df[[
    'country',
    'total_Population_million_people',
    'population_density',
    'population_rank',
    'population_coverage_perc',
    'geographic_coverage_perc',
    'year'
]]

In [ ]:
# Hypothetical Scenarios:
# 4. Population Data Conflict -exemple

import pandas as pd

data = [
    {'country': 'United States', 'year': 2020, 'population': 138_000_000, 'source': 'official', 'source_year': 2020},
    {'country': 'United States', 'year': 2020, 'population': 139_000_000, 'source': 'third_party', 'source_year': 2021},
    {'country': 'United States', 'year': 2021, 'population': 140_000_000, 'source': 'official', 'source_year': 2021},
    {'country': 'Germany', 'year': 2020, 'population': 83_000_000, 'source': 'official', 'source_year': 2020},
    {'country': 'Germany', 'year': 2020, 'population': 83_500_000, 'source': 'third_party', 'source_year': 2022},
]

df = pd.DataFrame(data)

# Group by country and year, keeping the record with the largest source_year (i.e. the most recent)
df_latest = df.sort_values(by='source_year', ascending=False) \
              .drop_duplicates(subset=['country', 'year'], keep='first')

df_latest = df_latest[['country', 'year', 'population']]

print(df_latest)





         country  year  population
4        Germany  2020    83500000
1  United States  2020   139000000
2  United States  2021   140000000


In [ ]:
# Hypothetical Scenarios:
# 5. Forest Coverage Conflict -exemple

import pandas as pd

# 模拟冲突数据：同一国家同一年来自不同来源的森林覆盖率
data = [
    {'country': 'Brazil', 'year': 2020, 'forest_area': 65.2, 'source': 'A'},
    {'country': 'Brazil', 'year': 2020, 'forest_area': 64.8, 'source': 'B'},
    {'country': 'Brazil', 'year': 2020, 'forest_area': 66.0, 'source': 'C'},
    {'country': 'Brazil', 'year': 2021, 'forest_area': 64.5, 'source': 'A'},
    {'country': 'Brazil', 'year': 2021, 'forest_area': 65.0, 'source': 'B'},
    {'country': 'Brazil', 'year': 2022, 'forest_area': 63.9, 'source': 'A'},
    {'country': 'Brazil', 'year': 2022, 'forest_area': 64.2, 'source': 'B'},
    {'country': 'Brazil', 'year': 2022, 'forest_area': 64.0, 'source': 'C'},
]

df = pd.DataFrame(data)

# 按国家 + 年份分组，计算平均值（即 Meet in the middle）
df_avg = df.groupby(['country', 'year'], as_index=False)['forest_area'].mean()

# 查看融合后的结果
print(df_avg)


In [ ]:
# Hypothetical Scenarios:
# 6. Air Quality Data Conflict -exemple